# oracle  —  Sec 3.3  selection from the 6000-key bank

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# Oracle control — right manifold, right *point*

The grid failed: snapping each token to the nearest on-manifold codebook point transferred 0.00, even though
fidelity improved. Was it (A) can't reach the manifold, or (B) on-manifold content is crap? This notebook
settles it with an **oracle**: for every token, inject the reader's **own genuine key/value for that token** —
provably on-manifold *and* the exact right point.

| arm | what it injects | tests |
|---|---|---|
| `identity` | holder cache **unchanged** | upper bound of KV injection; is the *path* fine? |
| `oracle_bycon` | `K=W_K·h`, `V=W_V·h` from the reader's **own captured h** | is the column-space construction lossless *at the true h*? |
| `bank:raw` | nearest real key from a 6k bank (a **different** token) | on-manifold, **wrong point** (grid contrast) |
| `knn_perturb` | oracle → moved toward its *k*-th nearest neighbour | **tolerance radius**: how far off-point before transfer dies |

Prediction: `identity`→ceiling, `oracle_bycon`→~ceiling (isolates bycon's point-finding failure from the
construction), `bank:raw`→floor, `knn_perturb`→decay from ceiling to floor mapping the razor-thin region.

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)
BASE="Qwen/Qwen2.5-1.5B"
CFG=dict(
  bank_ctx=120, bank_cap=6000,
  knn_ks=[1,2,4,8,16],       # distance-sweep neighbour ranks
  n_eval=200, max_ctx=256, gen_len=8,
  alphas=[0.0,0.1,0.25,0.5,0.75,1.0],
)
if os.environ.get("KVL_FAST","0")=="1":
    CFG.update(bank_ctx=30,bank_cap=1500,n_eval=24)
    print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_oraclefast" if os.environ.get("KVL_FAST","0")=="1" else "_oracle"
TAG=os.environ.get("KVL_TAG", _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)

## 1. Load frozen base; W_Q/W_K/W_V + RoPE (identical rig)

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,cache_dir=CACHE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
base=AutoModelForCausalLM.from_pretrained(BASE,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)
ids=tok("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): pkv=base(ids,use_cache=True).past_key_values
N_LAYER=len(pkv); _,N_KV,_,D_HEAD=pkv[0][0].shape
N_Q=base.config.num_attention_heads; GROUP=N_Q//N_KV; KV_DIM=2*N_KV*D_HEAD
KDIM=N_KV*D_HEAD; HID=base.config.hidden_size
layers=base.model.layers
WQ=[layers[l].self_attn.q_proj.weight.detach() for l in range(N_LAYER)]
BQ=[getattr(layers[l].self_attn.q_proj,'bias',None) for l in range(N_LAYER)]
WK=[layers[l].self_attn.k_proj.weight.detach() for l in range(N_LAYER)]
BK=[getattr(layers[l].self_attn.k_proj,'bias',None) for l in range(N_LAYER)]
WV=[layers[l].self_attn.v_proj.weight.detach() for l in range(N_LAYER)]
BV=[getattr(layers[l].self_attn.v_proj,'bias',None) for l in range(N_LAYER)]
print(f"L={N_LAYER} n_q={N_Q} n_kv={N_KV} group={GROUP} d_head={D_HEAD} kdim={KDIM} hid={HID}",flush=True)
_THETA=float(getattr(base.config,'rope_theta',1e6))
_INV=1.0/(_THETA**(torch.arange(0,D_HEAD,2,device=DEVICE).float()/D_HEAD))
def _cossin(T):
    pos=torch.arange(T,device=DEVICE).float(); ang=torch.outer(pos,_INV); emb=torch.cat([ang,ang],-1)
    return emb.cos()[None], emb.sin()[None]
def rope(x,T):
    cos,sin=_cossin(T); t1,t2=x[...,:D_HEAD//2],x[...,D_HEAD//2:]
    rr=torch.cat([-t2,t1],-1); return x*cos+rr*sin

## 2. Cache tooling + reserved-last-token decode (identical)

In [ ]:
def kv_stack(past):
    outs=[]
    for (k,v) in past:
        B,H,T,dh=k.shape
        outs.append(torch.cat([k.permute(0,2,1,3).reshape(B,T,H*dh),
                               v.permute(0,2,1,3).reshape(B,T,H*dh)],-1))
    return torch.stack(outs,0)
def native_ids(text,max_len):
    return tok([text],return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
@torch.no_grad()
def ctx_cache(text,max_len):
    ids=native_ids(text,max_len); last=ids[:,-1:]; body=ids[:,:-1]
    leg=base(body,use_cache=True).past_key_values
    if not isinstance(leg,tuple): leg=leg.to_legacy_cache() if hasattr(leg,"to_legacy_cache") else leg
    return kv_stack(leg), last, body
def slots_to_legacy(vec):
    L,B,Kk,_=vec.shape; half=KDIM; leg=[]
    for l in range(L):
        kt=vec[l,:,:,:half].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        vt=vec[l,:,:,half:].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        leg.append((kt,vt))
    return leg
def to_past(vec): return DynamicCache.from_legacy_cache(tuple(slots_to_legacy(vec)))
def _pos(s,l): return torch.arange(s,s+l,device=DEVICE).unsqueeze(0)
@torch.no_grad()
def greedy_from_prefix(prefix_vec,last_id):
    past=to_past(prefix_vec); T0=prefix_vec.shape[2]
    o=base(last_id,past_key_values=past,position_ids=_pos(T0,1),
           attention_mask=torch.ones(1,T0+1,device=DEVICE),use_cache=True)
    past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen=[cur]; L=T0+1
    for _ in range(CFG["gen_len"]-1):
        o=base(cur,past_key_values=past,position_ids=_pos(L,1),
               attention_mask=torch.ones(1,L+1,device=DEVICE),use_cache=True)
        past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen.append(cur); L+=1
    return tok.decode(torch.cat(gen,1)[0])

## 3. Split-context SQuAD (identical)

In [ ]:
def make_prompt(p,q): return f"{p}\nQuestion: {q}\nAnswer:"
def mask_answer(p,a):
    i=p.find(a); return None if i<0 else p[:i]+"____"+p[i+len(a):]
def build_qa(n,split):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for ex in ds:
        a=ex["answers"]["text"][0].strip()
        if not a or a not in ex["context"] or len(a.split())>4: continue
        m=mask_answer(ex["context"],a)
        if m is None: continue
        items.append(dict(q=ex["question"].strip(),full=ex["context"].strip(),masked=m.strip(),ans=a))
        if len(items)>=n: break
    return items
train_qa=build_qa(CFG["bank_ctx"],"train"); eval_qa=build_qa(CFG["n_eval"],"validation")
print("train",len(train_qa),"eval",len(eval_qa),flush=True)
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
def hit(t,g): return norm(g) in norm(t) if g else False
@torch.no_grad()
def holder_full(it): return ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
@torch.no_grad()
def holder_masked(it): return ctx_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
@torch.no_grad()
def reader_self(it): return ctx_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])

## 4. Capture genuine per-layer hidden Hs[l] (the true point's coordinates)

In [ ]:
@torch.no_grad()
def capture_h(body):
    Hs={}; hooks=[]
    def mk(l):
        def h(mod,inp,out): Hs[l]=out[0].detach()
        return h
    for l in range(N_LAYER): hooks.append(layers[l].input_layernorm.register_forward_hook(mk(l)))
    base(body,use_cache=False)
    for h in hooks: h.remove()
    return Hs

## 5. Build the real-key bank (for the `bank:raw` contrast + knn neighbours)
Same harvest as the grid: real K,V slots over `bank_ctx` contexts, capped per layer.

In [ ]:
@torch.no_grad()
def harvest():
    Kb=[[] for _ in range(N_LAYER)]; Vb=[[] for _ in range(N_LAYER)]
    for it in train_qa[:CFG["bank_ctx"]]:
        vec,_,_=holder_full(it)
        for l in range(N_LAYER):
            Kb[l].append(vec[l,0,:,:KDIM]); Vb[l].append(vec[l,0,:,KDIM:])
    def cat_cap(lst):
        out=[]
        for l in range(N_LAYER):
            X=torch.cat(lst[l],0)
            if X.shape[0]>CFG["bank_cap"]:
                idx=torch.randperm(X.shape[0],device=X.device)[:CFG["bank_cap"]]; X=X[idx]
            out.append(X.contiguous())
        return out
    return cat_cap(Kb),cat_cap(Vb)
Kbank,Vbank=harvest()
Kbank_n=[Kbank[l]/(Kbank[l].norm(dim=1,keepdim=True)+1e-9) for l in range(N_LAYER)]
print("bank sizes",[Kbank[l].shape[0] for l in range(0,N_LAYER,6)],flush=True)

## 6. The arms — pure injection, no training

- `identity`: return the holder cache unchanged.
- `oracle_bycon`: rebuild K,V from the reader's OWN captured hidden `h` via frozen `W_K/W_V`.
- `bank:raw`: nearest real bank key per token (grid contrast).
- `knn_perturb(K)`: from the true key, move a fraction toward its K-th nearest bank neighbour (per token).
  We report a full `alpha` blend so the tolerance radius is a curve, not a point.

In [ ]:
@torch.no_grad()
def kv_from_h(Hs):
    out=[]
    for l in range(N_LAYER):
        K=Hs[l]@WK[l].T + (BK[l] if BK[l] is not None else 0.0)
        V=Hs[l]@WV[l].T + (BV[l] if BV[l] is not None else 0.0)
        out.append(torch.cat([K,V],-1))
    return torch.stack(out,0).unsqueeze(1)   # (L,1,T,KV_DIM)

@torch.no_grad()
def bank_raw(vec):
    out=vec.clone()
    for l in range(N_LAYER):
        Ks=vec[l,0,:,:KDIM]; Kn=Ks/(Ks.norm(dim=1,keepdim=True)+1e-9)
        j=(Kn@Kbank_n[l].T).argmax(1)
        out[l,0,:,:KDIM]=Kbank[l][j]; out[l,0,:,KDIM:]=Vbank[l][j]
    return out

@torch.no_grad()
def knn_neighbour_targets(vec,krank):
    # for each token, the krank-th nearest DISTINCT bank key/value (a real on-manifold point, wrong token)
    tgtK=vec.clone();
    outK=vec[:, :, :, :KDIM].clone(); outV=vec[:, :, :, KDIM:].clone()
    for l in range(N_LAYER):
        Ks=vec[l,0,:,:KDIM]; Kn=Ks/(Ks.norm(dim=1,keepdim=True)+1e-9)
        sim=Kn@Kbank_n[l].T
        k=min(krank,Kbank[l].shape[0]-1)
        _,ti=sim.topk(k+1,dim=1)          # include self-ish; take the k-th ranked
        sel=ti[:,k]
        outK[l,0]=Kbank[l][sel]; outV[l,0]=Vbank[l][sel]
    return torch.cat([outK,outV],-1)

def apply_arm(arm,vec,Hs):
    if arm=="identity":     return vec
    if arm=="oracle_bycon": return kv_from_h(Hs)
    if arm=="bank:raw":     return bank_raw(vec)
    raise ValueError(arm)

## 7. Eval harness (transfer + per-layer fidelity + a-sweep), identical metrics

In [ ]:
@torch.no_grad()
def per_layer_manifold(applied_vec,vec,Hs_dummy,body):
    Hs=capture_h(body) if body is not None else None
    keys=["cos","logit_corr"]; res=[]
    scale=1.0/(D_HEAD**0.5); half=KDIM
    # need Q: recompute from captured h
    Q={}
    if Hs is not None:
        for l in range(N_LAYER):
            q=Hs[l]@WQ[l].T + (BQ[l] if BQ[l] is not None else 0.0)
            Q[l]=q.view(-1,N_Q,D_HEAD).permute(1,0,2)
    T=vec.shape[2]; mask=torch.triu(torch.ones(T,T,device=DEVICE),1).bool(); vld=~mask
    for l in range(N_LAYER):
        k =vec        [l,0,:,:half].reshape(T,N_KV,D_HEAD).permute(1,0,2)
        kh=applied_vec[l,0,:,:half].reshape(T,N_KV,D_HEAD).permute(1,0,2)
        c=F.cosine_similarity(kh.reshape(-1),k.reshape(-1),0).item()
        if Q:
            qr=rope(Q[l],T); kr=rope(k,T).repeat_interleave(GROUP,0); krh=rope(kh,T).repeat_interleave(GROUP,0)
            Lt=(torch.einsum('htd,hsd->hts',qr,kr)*scale).masked_fill(mask,float('-inf'))
            Lh=(torch.einsum('htd,hsd->hts',qr,krh)*scale).masked_fill(mask,float('-inf'))
            a=Lt[:,vld].reshape(-1); b=Lh[:,vld].reshape(-1); fin=torch.isfinite(a)&torch.isfinite(b)
            lc=torch.corrcoef(torch.stack([a[fin],b[fin]]))[0,1].item() if fin.sum()>2 else 0.0
        else: lc=0.0
        res.append(dict(cos=c,logit_corr=lc))
    return res

@torch.no_grad()
def fidelity(arm,items):
    acc=[[] for _ in range(N_LAYER)]; laccc=[[] for _ in range(N_LAYER)]
    for it in items[:min(len(items),60)]:
        vec,_,body=holder_full(it); Hs=capture_h(body)
        av=apply_arm(arm,vec,Hs)
        pl=per_layer_manifold(av,vec,None,body)
        for l in range(N_LAYER): acc[l].append(pl[l]["cos"]); laccc[l].append(pl[l]["logit_corr"])
    return [dict(cos=float(np.mean(acc[l])),logit_corr=float(np.mean(laccc[l]))) for l in range(N_LAYER)]

@torch.no_grad()
def eval_transfer(arm,items):
    A={"floor":0,"ceiling":0,"esrd_shared":0,"esrd_ctrl":0,"esrd_self":0}; n=0
    for it in items:
        g=it["ans"]
        v,l,_=reader_self(it);   A["floor"]  +=hit(greedy_from_prefix(v,l),g)
        v,l,_=holder_full(it);   A["ceiling"]+=hit(greedy_from_prefix(v,l),g)
        v,l,b=holder_full(it);   Hs=capture_h(b); A["esrd_shared"]+=hit(greedy_from_prefix(apply_arm(arm,v,Hs),l),g)
        v,l,b=holder_masked(it); Hs=capture_h(b); A["esrd_ctrl"]  +=hit(greedy_from_prefix(apply_arm(arm,v,Hs),l),g)
        v,l,b=reader_self(it);   Hs=capture_h(b); A["esrd_self"]  +=hit(greedy_from_prefix(apply_arm(arm,v,Hs),l),g)
        n+=1
    return {k:v/max(1,n) for k,v in A.items()}

@torch.no_grad()
def a_sweep_target(target_vec_fn,items):
    # blend own -> target; target_vec_fn(it)-> (target_vec, own_vec, last_id)
    out={f"a{a}":0 for a in CFG["alphas"]}; n=0
    for it in items:
        g=it["ans"]; own,last,_=reader_self(it); tgt=target_vec_fn(it)
        S=min(own.shape[2],tgt.shape[2])
        for a in CFG["alphas"]:
            c=own.clone(); c[:,:,:S,:]=(1-a)*own[:,:,:S,:]+a*tgt[:,:,:S,:]
            out[f"a{a}"]+=hit(greedy_from_prefix(c,last),g)
        n+=1
    return {k:v/max(1,n) for k,v in out.items()}

## 8. Run the arms + the knn distance sweep

For `knn_perturb` we run the a-sweep against each neighbour rank K: `own -> k-th-nearest-bank-key`. Reading
the a=1.0 endpoint across K gives transfer as a function of "how far off the exact point" (rank 1 = closest
wrong key, rank 16 = further). This is the tolerance-radius map.

In [ ]:
RES={"config":CFG,"base":BASE,"arms":{},"knn":{}}
RESPATH=f"results/oracle{TAG}.json"

@torch.no_grad()
def _sw_target(arm,it):
    vec,last,body=holder_full(it); Hs=capture_h(body)
    return apply_arm(arm,vec,Hs)

ARMS=["identity","oracle_bycon","bank:raw"]
for arm in ARMS:
    print(f"\n########## ARM {arm} ##########",flush=True)
    tr=eval_transfer(arm,eval_qa)
    pl=fidelity(arm,eval_qa)
    mc=float(np.mean([d["cos"] for d in pl])); mlc=float(np.mean([d["logit_corr"] for d in pl]))
    nov=int(sum(d["logit_corr"]>0.99 for d in pl))
    sw=a_sweep_target(lambda it,_arm=arm: _sw_target(_arm,it), eval_qa)
    RES["arms"][arm]=dict(transfer=tr,per_layer=pl,mean_cos=mc,mean_logit_corr=mlc,n_over_wall=nov,sweep_acc=sw)
    print(f"[{arm}] floor={tr['floor']:.3f} ceil={tr['ceiling']:.3f} shared={tr['esrd_shared']:.3f} "
          f"ctrl={tr['esrd_ctrl']:.3f} self={tr['esrd_self']:.3f}",flush=True)
    print(f"[{arm}] cos={mc:.3f} lc={mlc:.3f} wall={nov}/{N_LAYER}",flush=True)
    print(f"[{arm}] a-sweep "+" ".join(f"a{a}={sw[f'a{a}']:.3f}" for a in CFG["alphas"]),flush=True)
    json.dump(RES,open(RESPATH,"w"),indent=1)

## 9. knn distance sweep — tolerance radius

In [ ]:
@torch.no_grad()
def eval_transfer_knn(krank,items):
    A={"esrd_shared":0}; n=0
    for it in items:
        g=it["ans"]; v,l,b=holder_full(it)
        tgt=knn_neighbour_targets(v,krank)
        A["esrd_shared"]+=hit(greedy_from_prefix(tgt,l),g); n+=1
    return A["esrd_shared"]/max(1,n)

for K in CFG["knn_ks"]:
    acc=eval_transfer_knn(K,eval_qa)
    sw=a_sweep_target(lambda it,KK=K: knn_neighbour_targets(holder_full(it)[0],KK), eval_qa)
    RES["knn"][f"k{K}"]=dict(full_sub_acc=acc,sweep_acc=sw)
    print(f"[knn k={K}] full-substitution shared={acc:.3f} | a-sweep "
          +" ".join(f"a{a}={sw[f'a{a}']:.3f}" for a in CFG["alphas"]),flush=True)
    json.dump(RES,open(RESPATH,"w"),indent=1)

## 10. Verdict

In [ ]:
fl=RES["arms"]["identity"]["transfer"]["floor"]; ce=RES["arms"]["identity"]["transfer"]["ceiling"]
print(f"\n=== ORACLE VERDICT (floor={fl:.3f} ceiling={ce:.3f}) ===",flush=True)
for arm in ARMS:
    a=RES["arms"][arm]; t=a["transfer"]
    flag=" <-- BEATS FLOOR" if t["esrd_shared"]>t["floor"]+1e-9 else ""
    print(f"  {arm:14s}: cos {a['mean_cos']:.3f} lc {a['mean_logit_corr']:.3f} wall {a['n_over_wall']:2d}/{N_LAYER} "
          f"shared {t['esrd_shared']:.3f} ctrl {t['esrd_ctrl']:.3f}{flag}",flush=True)
print("  --- tolerance radius (full substitution to k-th nearest bank key) ---",flush=True)
for K in CFG["knn_ks"]:
    print(f"  knn k={K:2d}: shared {RES['knn'][f'k{K}']['full_sub_acc']:.3f}",flush=True)
for _line in [
 "INTERPRETATION:",
 " * identity >> floor  => KV injection CAN transfer; path is fine; the grid merely missed the point.",
 " * oracle_bycon ~ identity => column-space construction is lossless AT THE TRUE h => bycon's wall was",
 "     point-FINDING (decoder couldn't produce the true h), not the construction.",
 " * bank:raw ~ floor, knn decays ceiling(k small)->floor(k large) => reader tolerates only a razor-thin",
 "     neighbourhood of the EXACT point => on-manifold is necessary but nowhere near sufficient; you need",
 "     the exact per-token point, which no finite codebook resolves. NEITHER (A) nor (B): on-manifold good",
 "     content IS reachable, but the tolerable target is a single point, not a set.",
]: print(_line,flush=True)
json.dump(RES,open(RESPATH,"w"),indent=1); print("saved",RESPATH,flush=True)

## 11. Plot — oracle bars + tolerance-radius curve

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4.4))
labels=ARMS+["floor","ceiling"]
vals=[RES["arms"][a]["transfer"]["esrd_shared"] for a in ARMS]+[fl,ce]
cols=["#2c7fb8","#41ab5d","#d95f0e","#999999","#000000"]
ax[0].bar(range(len(labels)),vals,color=cols)
ax[0].set_xticks(range(len(labels))); ax[0].set_xticklabels(labels,rotation=30,ha="right")
ax[0].set_title("transfer (esrd_shared)"); ax[0].axhline(fl,ls='--',color='#999'); ax[0].axhline(ce,ls='--',color='#000')
for i,v in enumerate(vals): ax[0].text(i,v+.01,f"{v:.2f}",ha="center",fontsize=9)
# tolerance radius
ks=CFG["knn_ks"]; ys=[RES["knn"][f"k{K}"]["full_sub_acc"] for K in ks]
ax[1].plot(ks,ys,'-o',color="#d95f0e"); ax[1].axhline(fl,ls='--',color='#999',label="floor")
ax[1].axhline(ce,ls='--',color='#000',label="ceiling"); ax[1].set_xscale("log",base=2)
ax[1].set_xlabel("k-th nearest bank key (distance off exact point)"); ax[1].set_title("tolerance radius"); ax[1].legend()
# a-sweeps: identity/oracle_bycon/bank:raw
for arm,c in zip(ARMS,cols):
    xs=CFG["alphas"]; ax[2].plot(xs,[RES["arms"][arm]["sweep_acc"][f"a{x}"] for x in xs],'-o',label=arm,color=c)
ax[2].axhline(fl,ls='--',color='#999'); ax[2].set_title("a-sweep (0=own,1=arm)"); ax[2].legend(fontsize=8)
for a_ in ax: a_.grid(ls=':',alpha=.4)
fig.tight_layout(); fig.savefig("figures/oracle_curves.png",dpi=150); plt.show()